In [1]:
from dotenv  import load_dotenv
from langchain_groq import ChatGroq 
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [5]:
PRODUCTS  = {
    "Wireless":{"Price": 79.99, "description": "Over-ear Bluetooth, 30-hr Battery, active noise cancellation."},
    "smart-watch":{"Price": 199.99, "description": "Tracks heart rate and sleep. 5-Day battery, water resistent."},
    "Gaming-Keyboard":{"Price": 200.00, "description": "Tenkey less, Cherry MX Brown, per key RGB."},
    "Speakers": {"Price": 249.99, "description": "Best Audionic Speaker, Bass-Boost Sound, RGB-lights."}
}

PRODUCTS_LOOKUP = {k.lower(): v for k, v in PRODUCTS.items()}

@tool
def get_product(name:str) -> str:
    """ Look up a product by name and return its price, rating, stock and description."""
    p = PRODUCTS_LOOKUP.get(name.lower())
    if not p:
        return f"product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
   system_prompt = (
    "You are a product assistant for an online tech store. "
    "Always call the get_product tool with the user's best-guess product name — "
    "do not ask the user to confirm the name before calling the tool. "
    "The tool performs the lookup and will tell you if the product isn't found, "
    "along with a list of available products. "
    "Only ask the user for clarification if the tool returns a 'not found' result, "
    "and use the available-products list from the tool's response to suggest options. "
    "Never guess or make up prices, descriptions, or availability — always rely on the tool's output."
))

In [12]:
# def ask(question: str):
#     result = agent.invoke({"messages": [{"role":"user", "content": question}]})
#     print(result["messages"] [-1].content)

In [7]:
def chat_loop():
    history = []
    print("Ask me about our products! (type 'exit', or 'quit' to end)\n")
    while True:
        question = input("You: ").strip()
        if question.lower() in ("exit", "quit"):
            print("Agent: Goodbye!")
            break
        if not question:
            continue
        history.append({"role": "user", "content": question})
        result = agent.invoke({"messages": history})
        reply = result["messages"][-1].content
        history.append({"role": "assistant", "content": reply})
        print("Agent:", reply, "\n")

chat_loop()

Ask me about our products! (type 'exit', or 'quit' to end)



You:  describe the product speakers


Agent: Here’s a quick overview of the **Speakers** you asked about:

- **Price:** $249.99  
- **Description:** Best Audionic Speaker, Bass‑Boost Sound, RGB‑lights  

These speakers are marketed as a premium audio solution with enhanced bass performance and customizable RGB lighting for a stylish look. If you’d like more details—such as availability, customer ratings, or specifications—just let me know! 



You:  list of products


Agent: Here are the product categories currently available in our catalog:

- **Wireless**
- **Smart‑watch**
- **Gaming‑Keyboard**
- **Speakers**

Let me know which one you’d like more details about, and I’ll pull up the full information for you! 



You:  quit


Agent: Goodbye!


In [13]:
ask(input("Ask the question"))

Ask the question what is the price of the speaker


I’m sorry—I couldn’t find a product listed exactly as “speaker” in our catalog. The closest match we have is a product called **Speakers**. Would you like me to give you the price (and other details) for the Speakers, or is there another item you had in mind?


In [4]:
print(get_product.invoke({"name": "Speakers"}))

{'Price': 249.99, 'description': 'Best Audionic Speaker, Bass-Boost Sound, RGB-lights.'}
